# Capa Gold — Agregaciones para análisis

Genera tablas analíticas optimizadas a partir de los datos Silver. El resultado es una tabla de hechos agregados lista para BI.

**Prerequisito:** haber ejecutado `01_Bronze_Ingesta` y `02_Silver_Limpieza_Enriquecimiento`.

In [0]:
# Leés el valor que mandó Data Factory:
storage_key = dbutils.widgets.get("storage_key")

# Validás que no venga vacío
if not storage_key:
    raise ValueError("El parámetro storage_key no fue provisto.")

## Paso 1 — Configurar acceso al Storage Account

In [0]:
# Acceso directo al storage (sin mount, compatible con clusters Serverless)
storage_account = "icarostorage" # Reemplazar con tu storage
container       = "tpf-medallion-data" #Crear contenedor para medallion
key             = storage_key

spark.conf.set(
    f"fs.azure.account.key.{storage_account}.blob.core.windows.net",
    key
)

BASE = f"wasbs://{container}@{storage_account}.blob.core.windows.net"

# Verificar que los archivos son accesibles
dbutils.fs.ls(BASE)

## Paso 2 — Leer la tabla Silver

In [0]:
from pyspark.sql.functions import sum, col, round, count, max, min, avg, when

# Leer la tabla Silver de ventas enriquecidas
pokemon_silver = spark.read.format("delta").load(f"{BASE}/silver/pokemon_silver")

print("Filas en Silver:", pokemon_silver.count())
pokemon_silver.printSchema()


## Paso 3 — Agregar: ventas totales por fecha y tienda

Agrupamos por `fecha` y `nombre_tienda` (no por ID, para que la tabla Gold sea autoexplicativa) y calculamos el total de unidades vendidas y el monto total.

In [0]:
# KPIs por tipo primario (Capa Gold)
gold_df = pokemon_silver \
    .groupBy("primary_type") \
    .agg(
        count("pokemon_id").alias("total_pokemon"),
        round(avg("base_stat_total"), 2).alias("avg_base_stats"),
        max("base_stat_total").alias("max_base_stats"),
        min("base_stat_total").alias("min_base_stats"),
        sum(when(col("is_starter") == True, 1).otherwise(0)).alias("starter_count")
    ) \
    .orderBy(col("avg_base_stats").desc())

gold_df.show()


## Paso 4 — Guardar la tabla Gold en Delta

In [0]:
# Persistir la tabla Gold en el blob storage
gold_df.write.format("delta").mode("overwrite").save(f"{BASE}/gold/pokemon_por_tipo")

print("Tabla Gold guardada en:", f"{BASE}/gold/pokemon_por_tipo")


## Paso 5 — Consulta SQL de validación

Registramos la tabla como vista temporal para poder consultarla con SQL.

In [0]:
# Registrar como vista temporal para SQL
gold_df.createOrReplaceTempView("gold_pokemon_por_tipo")


In [0]:
%sql
-- Top 10 días con más ventas totales
SELECT primary_type, total_pokemon, avg_base_stats, max_base_stats, min_base_stats, starter_count
FROM gold_pokemon_por_tipo
ORDER BY total_pokemon DESC
LIMIT 10;


In [0]:
display(gold_df)

Databricks visualization. Run in Databricks to view.